# 📚 E-Commerce Sales & Pricing Analyzer
### End-to-end Data Analytics Project
**Tools:** Python · Selenium · pandas · MySQL · Excel (openpyxl) · Tableau

---

**Business Question:**  
*Which product categories are priced competitively, and where should we focus next quarter?*

**Data Source:** [books.toscrape.com](http://books.toscrape.com) — a public practice site designed for scraping projects.

---

### Pipeline Overview

| Phase | Tool | Task |
|-------|------|------|
| 1 | Python + Selenium | Scrape product data from the web |
| 2 | Python + pandas | Clean and transform raw data |
| 3 | Excel (openpyxl) | Build pivot tables, charts & KPI report |
| 4 | MySQL | Store data and answer business questions with SQL |
| 5 | Tableau | Export data for interactive dashboard |


## ⚙️ Setup

Install all required packages and create the `data/` directory.

In [ ]:
%pip install selenium webdriver-manager pandas mysql-connector-python openpyxl

In [ ]:
import os
import time
import logging
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Create data directory
os.makedirs("data", exist_ok=True)
print("✅ Setup complete. data/ directory ready.")

---
## Phase 1 — Web Scraping with Selenium

We scrape product listings (title, price, star rating, availability) from **books.toscrape.com** across 10 pages (~200 books).

> **Note:** This site is purpose-built for scraping practice — no real data is harmed. `headless=True` runs Chrome in the background without opening a window.


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

BASE_URL    = "http://books.toscrape.com/catalogue/page-{}.html"
TOTAL_PAGES = 10  # ~200 books total

def init_driver(headless=True):
    """Initialize Chrome WebDriver."""
    options = Options()
    if headless:
        options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1280,800")
    return webdriver.Chrome(options=options)

def scrape_page(driver, page_num):
    """Scrape one page — returns a list of product dicts."""
    driver.get(BASE_URL.format(page_num))
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )
    except Exception as e:
        print(f"⚠️  Page {page_num} timeout: {e}")
        return []

    products = []
    for book in driver.find_elements(By.CLASS_NAME, "product_pod"):
        try:
            title        = book.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
            price_text   = book.find_element(By.CLASS_NAME, "price_color").text
            rating_class = book.find_element(By.CLASS_NAME, "star-rating").get_attribute("class")
            availability = book.find_element(By.CLASS_NAME, "availability").text.strip()
            products.append({
                "title":        title,
                "price_raw":    price_text,
                "rating_raw":   rating_class,
                "availability": availability,
                "page":         page_num,
            })
        except Exception as e:
            print(f"⚠️  Error on page {page_num}: {e}")
    return products

print("✅ Scraper functions defined.")

In [ ]:
# ── Run the scraper ──────────────────────────────────────────────────────────
print("🚀 Starting scraper...")
driver      = init_driver(headless=True)
all_products = []

try:
    for page in range(1, TOTAL_PAGES + 1):
        products = scrape_page(driver, page)
        all_products.extend(products)
        print(f"  Page {page:>2}: scraped {len(products)} products  (total so far: {len(all_products)})")
        time.sleep(1)  # polite delay
finally:
    driver.quit()

df_raw = pd.DataFrame(all_products)
df_raw.to_csv("data/raw_products.csv", index=False)
print(f"\n✅ Saved {len(df_raw)} rows → data/raw_products.csv")
df_raw.head(5)

In [ ]:
# Quick peek at raw data
print(f"Shape: {df_raw.shape}")
print(f"\nColumn dtypes:\n{df_raw.dtypes}")
print(f"\nSample price values:  {df_raw['price_raw'].unique()[:5]}")
print(f"Sample rating values: {df_raw['rating_raw'].unique()[:5]}")

---
## Phase 2 — Data Cleaning & Transformation with pandas

Raw scraped data needs significant tidying before it's usable:
- **Price**: strip currency symbols → float
- **Rating**: map text class names → integers 1–5
- **Availability**: convert to binary `in_stock` flag
- **Price tier**: bin into 4 labelled categories for grouping
- **Duplicates**: detect and remove


In [ ]:
RATING_MAP = {
    "star-rating One":   1,
    "star-rating Two":   2,
    "star-rating Three": 3,
    "star-rating Four":  4,
    "star-rating Five":  5,
}

PRICE_BINS   = [0, 15, 35, 60, 100]
PRICE_LABELS = ["Budget (£0-15)", "Mid (£15-35)", "Upper-Mid (£35-60)", "Premium (£60+)"]

def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Price: strip non-numeric chars
    df["price_gbp"] = (
        df["price_raw"]
        .str.replace(r"[^\d.]", "", regex=True)
        .astype(float)
    )

    # Rating: text class → integer
    df["rating_num"] = df["rating_raw"].map(RATING_MAP)
    missing = df["rating_num"].isna().sum()
    if missing:
        print(f"⚠️  {missing} unrecognised ratings — filling with median")
        df["rating_num"] = df["rating_num"].fillna(df["rating_num"].median())
    df["rating_num"] = df["rating_num"].astype(int)

    # Availability → binary flag
    df["in_stock"] = df["availability"].str.contains("In stock", case=False, na=False).astype(int)

    # Price tier (binning)
    df["price_tier"] = pd.cut(
        df["price_gbp"], bins=PRICE_BINS, labels=PRICE_LABELS, right=False
    ).astype(str)

    # Title cleanup
    df["title"] = df["title"].str.strip()

    # Drop raw columns
    df.drop(columns=["price_raw", "rating_raw", "availability"], inplace=True)

    # Deduplicate
    dupes = df.duplicated(subset="title").sum()
    if dupes:
        print(f"⚠️  Dropping {dupes} duplicate titles")
        df.drop_duplicates(subset="title", inplace=True)

    df.reset_index(drop=True, inplace=True)
    return df

print("✅ Clean function defined.")

In [ ]:
df_clean = clean(df_raw)
df_clean.to_csv("data/clean_products.csv", index=False)
print(f"✅ Saved {len(df_clean)} rows → data/clean_products.csv\n")
df_clean.head(8)

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
print("── Numeric summary ───────────────────────────────")
display(df_clean[["price_gbp", "rating_num", "in_stock"]].describe().round(2))

print("\n── Price tier distribution ───────────────────────")
display(df_clean["price_tier"].value_counts().rename("count").to_frame())

print("\n── Rating distribution ───────────────────────────")
display(df_clean["rating_num"].value_counts().sort_index().rename("count").to_frame())

---
## Phase 3 — Excel Report with openpyxl

We programmatically generate a formatted `.xlsx` workbook with four sheets:
1. **Summary** — KPI cards (total products, avg price, avg rating, stock count)
2. **Pivot — Price Tier** — grouped stats + bar chart
3. **Pivot — Rating** — grouped stats + conditional formatting (5-star rows highlighted green)
4. **Raw Data** — full dataset with frozen header and colour-coded rating column


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, Reference
from openpyxl.formatting.rule import CellIsRule
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

# ── Styles ────────────────────────────────────────────────────────────────────
HEADER_FILL  = PatternFill("solid", fgColor="1F4E79")
ACCENT_FILL  = PatternFill("solid", fgColor="D6E4F0")
GREEN_FILL   = PatternFill("solid", fgColor="E2EFDA")
ORANGE_FILL  = PatternFill("solid", fgColor="FCE4D6")
KPI_FILL     = PatternFill("solid", fgColor="EBF3FB")

WHITE_FONT   = Font(name="Calibri", bold=True,  color="FFFFFF", size=11)
BOLD_FONT    = Font(name="Calibri", bold=True,  size=11)
BODY_FONT    = Font(name="Calibri",              size=11)
TITLE_FONT   = Font(name="Calibri", bold=True,  size=14, color="1F4E79")
KPI_VAL_FONT = Font(name="Calibri", bold=True,  size=18, color="1F4E79")
KPI_LBL_FONT = Font(name="Calibri",              size=10, color="595959")

THIN   = Side(style="thin", color="BFBFBF")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT   = Alignment(horizontal="left",   vertical="center", wrap_text=True)

def style_header_row(ws, row, col_start, col_end):
    for col in range(col_start, col_end + 1):
        cell = ws.cell(row=row, column=col)
        cell.fill = HEADER_FILL;  cell.font = WHITE_FONT
        cell.alignment = CENTER;  cell.border = BORDER

def auto_width(ws, min_w=10, max_w=50):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=0)
        ws.column_dimensions[col[0].column_letter].width = min(max(w + 2, min_w), max_w)

print("✅ Styles defined.")

In [ ]:
def write_summary(wb, df):
    ws = wb.create_sheet("Summary")
    ws.sheet_view.showGridLines = False
    ws["B2"] = "E-Commerce Product Analysis";  ws["B2"].font = TITLE_FONT
    ws["B3"] = "Catalogue summary — books.toscrape.com"
    ws["B3"].font = Font(name="Calibri", size=11, color="595959", italic=True)

    kpis = [
        ("Total Products",  len(df)),
        ("Avg Price (£)",   f"£{df['price_gbp'].mean():.2f}"),
        ("Avg Rating",      f"{df['rating_num'].mean():.2f} / 5"),
        ("In Stock",        int(df['in_stock'].sum())),
        ("Premium Books",   int((df['price_tier'].str.startswith("Premium")).sum())),
        ("5-Star Books",    int((df['rating_num'] == 5).sum())),
    ]
    for i, (label, value) in enumerate(kpis):
        col = (i % 3) * 3 + 2;  row = 5 + (i // 3) * 4
        for r in range(row, row + 3):
            for c in range(col, col + 2):
                ws.cell(r, c).fill = KPI_FILL
        ws.cell(row,     col).value = label;  ws.cell(row,     col).font = KPI_LBL_FONT
        ws.cell(row + 1, col).value = value;  ws.cell(row + 1, col).font = KPI_VAL_FONT
        ws.merge_cells(start_row=row+1, start_column=col, end_row=row+1, end_column=col+1)
    ws.column_dimensions["A"].width = 2
    for c in ["B","C","D","E","F","G","H","I"]:
        ws.column_dimensions[c].width = 18

def write_tier_pivot(wb, df):
    ws = wb.create_sheet("Pivot - Price Tier")
    ws.sheet_view.showGridLines = False
    pivot = df.groupby("price_tier").agg(
        total_products=("title","count"), avg_price=("price_gbp","mean"),
        avg_rating=("rating_num","mean"), in_stock=("in_stock","sum")
    ).reset_index()
    pivot[["avg_price","avg_rating"]] = pivot[["avg_price","avg_rating"]].round(2)
    headers = ["Price Tier","Total Products","Avg Price (£)","Avg Rating","In Stock"]
    ws.append(headers);  style_header_row(ws, 1, 1, len(headers))
    for _, row in pivot.iterrows():
        ws.append(list(row))
    for ri in range(2, ws.max_row + 1):
        fill = ACCENT_FILL if ri % 2 == 0 else PatternFill()
        for ci in range(1, len(headers) + 1):
            cell = ws.cell(ri, ci)
            if fill.fill_type: cell.fill = fill
            cell.border = BORDER;  cell.font = BODY_FONT;  cell.alignment = CENTER
    chart = BarChart()
    chart.type = "col";  chart.title = "Products per Price Tier"
    chart.y_axis.title = "Count";  chart.x_axis.title = "Tier"
    chart.style = 10;  chart.width = 16;  chart.height = 12
    chart.add_data(Reference(ws, min_col=2, min_row=1, max_row=ws.max_row), titles_from_data=True)
    chart.set_categories(Reference(ws, min_col=1, min_row=2, max_row=ws.max_row))
    ws.add_chart(chart, "G2");  auto_width(ws)

def write_rating_pivot(wb, df):
    ws = wb.create_sheet("Pivot - Rating")
    ws.sheet_view.showGridLines = False
    pivot = df.groupby("rating_num").agg(
        count=("title","count"), avg_price=("price_gbp","mean"),
        min_price=("price_gbp","min"), max_price=("price_gbp","max")
    ).reset_index().rename(columns={"rating_num":"stars"}).sort_values("stars", ascending=False)
    for col in ["avg_price","min_price","max_price"]:
        pivot[col] = pivot[col].round(2)
    headers = ["Stars","Count","Avg Price (£)","Min Price (£)","Max Price (£)"]
    ws.append(headers);  style_header_row(ws, 1, 1, len(headers))
    for _, row in pivot.iterrows():
        ws.append(list(row))
    for ri in range(2, ws.max_row + 1):
        for ci in range(1, len(headers) + 1):
            cell = ws.cell(ri, ci)
            cell.border = BORDER;  cell.font = BODY_FONT;  cell.alignment = CENTER
    ws.conditional_formatting.add(
        f"A2:E{ws.max_row}",
        CellIsRule(operator="equal", formula=["5"], fill=GREEN_FILL)
    )
    auto_width(ws)

def write_raw_data(wb, df):
    ws = wb.create_sheet("Raw Data")
    headers = list(df.columns)
    ws.append(headers);  style_header_row(ws, 1, 1, len(headers))
    for row in dataframe_to_rows(df, index=False, header=False):
        ws.append(row)
    rc = headers.index("rating_num") + 1
    cl = get_column_letter(rc);  rng = f"{cl}2:{cl}{len(df)+1}"
    ws.conditional_formatting.add(rng, CellIsRule(operator="greaterThanOrEqual", formula=["4"], fill=GREEN_FILL))
    ws.conditional_formatting.add(rng, CellIsRule(operator="lessThanOrEqual",    formula=["2"], fill=ORANGE_FILL))
    ws.freeze_panes = "A2"
    for ri in range(2, ws.max_row + 1):
        for ci in range(1, len(headers) + 1):
            ws.cell(ri, ci).font = BODY_FONT;  ws.cell(ri, ci).alignment = LEFT;  ws.cell(ri, ci).border = BORDER
    auto_width(ws)

print("✅ Sheet builder functions defined.")

In [ ]:
# ── Build and save the workbook ────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

write_summary(wb, df_clean)
write_tier_pivot(wb, df_clean)
write_rating_pivot(wb, df_clean)
write_raw_data(wb, df_clean)

wb.save("data/excel_report.xlsx")
print("✅ Excel report saved → data/excel_report.xlsx")
print(f"Sheets: {[s.title for s in wb.worksheets]}")

---
## Phase 4 — MySQL: Database Setup & Business Queries

### 4a — Configuration

> ⚠️ **Update the password below** before running this section.


In [ ]:
import mysql.connector

# ── ✏️  UPDATE THESE WITH YOUR CREDENTIALS ────────────────────────────────────
DB_CONFIG = {
    "host":     "localhost",
    "port":     3306,
    "user":     "root",
    "password": "your_password",   # <── change this
}
DATABASE_NAME = "ecommerce_db"
# ──────────────────────────────────────────────────────────────────────────────

def get_conn(database=None):
    cfg = dict(DB_CONFIG)
    if database:
        cfg["database"] = database
    return mysql.connector.connect(**cfg)

print("✅ DB config ready.")

### 4b — Create Database & Table

In [ ]:
CREATE_DB_SQL = f"CREATE DATABASE IF NOT EXISTS `{DATABASE_NAME}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;"

CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS products (
    id          INT            AUTO_INCREMENT PRIMARY KEY,
    title       VARCHAR(500)   NOT NULL,
    price_gbp   DECIMAL(8,2)  NOT NULL,
    rating_num  TINYINT        NOT NULL CHECK (rating_num BETWEEN 1 AND 5),
    in_stock    TINYINT(1)     NOT NULL DEFAULT 1,
    price_tier  VARCHAR(50)    NOT NULL,
    page        SMALLINT,
    created_at  TIMESTAMP      DEFAULT CURRENT_TIMESTAMP
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""

# Create DB
conn   = get_conn()
cursor = conn.cursor()
cursor.execute(CREATE_DB_SQL)
cursor.close();  conn.close()
print(f"✅ Database '{DATABASE_NAME}' ready.")

# Create table
conn   = get_conn(database=DATABASE_NAME)
cursor = conn.cursor()
cursor.execute(CREATE_TABLE_SQL)
conn.commit();  cursor.close();  conn.close()
print("✅ Table 'products' ready.")

### 4c — Load Data into MySQL

In [ ]:
INSERT_SQL = """
INSERT INTO products (title, price_gbp, rating_num, in_stock, price_tier, page)
VALUES (%s, %s, %s, %s, %s, %s)
"""

# Truncate first so cell is safely re-runnable
conn   = get_conn(database=DATABASE_NAME)
cursor = conn.cursor()
cursor.execute("TRUNCATE TABLE products;")
conn.commit()

# Build rows list
df_load = pd.read_csv("data/clean_products.csv")
rows = [
    (
        row["title"],
        float(row["price_gbp"]),
        int(row["rating_num"]),
        int(row["in_stock"]),
        str(row["price_tier"]),
        int(row["page"]) if "page" in row and pd.notna(row["page"]) else None,
    )
    for _, row in df_load.iterrows()
]

cursor.executemany(INSERT_SQL, rows)
conn.commit()
print(f"✅ Inserted {cursor.rowcount} rows into MySQL.")
cursor.close();  conn.close()

### 4d — Verify Load

In [ ]:
conn   = get_conn(database=DATABASE_NAME)
cursor = conn.cursor(dictionary=True)
cursor.execute("SELECT COUNT(*) AS total FROM products;")
print(f"Total rows in DB: {cursor.fetchone()['total']}\n")
cursor.execute("SELECT * FROM products LIMIT 5;")
sample = pd.DataFrame(cursor.fetchall())
cursor.close();  conn.close()
display(sample)

### 4e — Business Queries

In [ ]:
def run_query(sql, label=""):
    """Run a SQL query and return a DataFrame."""
    conn = get_conn(database=DATABASE_NAME)
    df   = pd.read_sql(sql, conn)
    conn.close()
    if label:
        print(f"\n{'─'*50}")
        print(f" {label}")
        print(f"{'─'*50}")
        display(df)
    return df

print("✅ run_query() helper ready.")

In [ ]:
# Q1 — Which price tier has the highest average rating?
q1 = run_query("""
    SELECT
        price_tier,
        COUNT(*)                        AS total_products,
        ROUND(AVG(price_gbp), 2)        AS avg_price,
        ROUND(AVG(rating_num), 2)       AS avg_rating,
        SUM(in_stock)                   AS in_stock_count
    FROM products
    GROUP BY price_tier
    ORDER BY avg_rating DESC
""", "Q1 — Price tier performance")

In [ ]:
# Q2 — Top 20 best-value books (high rating, low price)
q2 = run_query("""
    SELECT
        title,
        price_gbp,
        rating_num,
        ROUND(rating_num / price_gbp, 4) AS value_score
    FROM products
    WHERE rating_num >= 4
    ORDER BY value_score DESC
    LIMIT 20
""", "Q2 — Best-value books (rating ÷ price)")

In [ ]:
# Q3 — Price distribution by star rating
q3 = run_query("""
    SELECT
        rating_num,
        COUNT(*)                  AS product_count,
        ROUND(MIN(price_gbp), 2)  AS min_price,
        ROUND(MAX(price_gbp), 2)  AS max_price,
        ROUND(AVG(price_gbp), 2)  AS avg_price
    FROM products
    GROUP BY rating_num
    ORDER BY rating_num DESC
""", "Q3 — Price distribution by rating")

In [ ]:
# Q4 — Stock availability by price tier
q4 = run_query("""
    SELECT
        price_tier,
        SUM(in_stock)                            AS in_stock,
        COUNT(*) - SUM(in_stock)                 AS out_of_stock,
        ROUND(SUM(in_stock) / COUNT(*) * 100, 1) AS availability_pct
    FROM products
    GROUP BY price_tier
    ORDER BY availability_pct ASC
""", "Q4 — Stock availability by tier")

In [ ]:
# Q5 — Top 10 most expensive in-stock books
q5 = run_query("""
    SELECT title, price_gbp, rating_num, price_tier
    FROM products
    WHERE in_stock = 1
    ORDER BY price_gbp DESC
    LIMIT 10
""", "Q5 — Top 10 premium in-stock books")

In [ ]:
# Q6 — Rating breakdown (% share of catalogue)
q6 = run_query("""
    SELECT
        rating_num AS stars,
        COUNT(*)   AS count,
        ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct_of_total
    FROM products
    GROUP BY rating_num
    ORDER BY rating_num DESC
""", "Q6 — Rating distribution (% of catalogue)")

In [ ]:
# Q7 — High-price, low-rating products (markdown candidates)
q7 = run_query("""
    SELECT title, price_gbp, rating_num, price_tier
    FROM products
    WHERE price_gbp > 40 AND rating_num <= 2
    ORDER BY price_gbp DESC
""", "Q7 — Expensive books with poor ratings (markdown candidates)")

In [ ]:
# Q10 — Executive summary
q10 = run_query("""
    SELECT
        COUNT(*)                             AS total_books,
        ROUND(AVG(price_gbp), 2)             AS avg_price_gbp,
        ROUND(AVG(rating_num), 2)            AS avg_rating,
        SUM(in_stock)                        AS in_stock_count,
        COUNT(*) - SUM(in_stock)             AS out_of_stock_count,
        ROUND(SUM(in_stock)/COUNT(*)*100, 1) AS availability_pct,
        MAX(price_gbp)                       AS most_expensive,
        MIN(price_gbp)                       AS cheapest
    FROM products
""", "Q10 — Executive catalogue summary")

### 4f — Export query results to Excel (one sheet per query)

In [ ]:
query_results = {
    "Q1_Tier_Performance":  q1,
    "Q2_Best_Value":        q2,
    "Q3_Price_By_Rating":   q3,
    "Q4_Stock_By_Tier":     q4,
    "Q5_Top_Premium":       q5,
    "Q6_Rating_Distribution": q6,
    "Q7_Underperformers":   q7,
    "Q10_Exec_Summary":     q10,
}

with pd.ExcelWriter("data/query_results.xlsx", engine="openpyxl") as writer:
    for sheet_name, df_q in query_results.items():
        df_q.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print("✅ Exported all query results → data/query_results.xlsx")
print(f"Sheets: {list(query_results.keys())}")

---
## Phase 5 — Tableau Export

Export analysis-ready CSVs from MySQL for use in Tableau (or Power BI).

> **Alternative:** In Tableau Desktop, go to **Connect → To a Server → MySQL** and enter `localhost`, your username, password, and `ecommerce_db`. This skips the CSV step entirely.


In [ ]:
tableau_queries = {
    "data/tableau_products.csv": """
        SELECT id, title, price_gbp, rating_num, in_stock, price_tier, page,
               ROUND(rating_num / price_gbp, 4) AS value_score
        FROM products ORDER BY id
    """,
    "data/tableau_tier_summary.csv": """
        SELECT
            price_tier,
            COUNT(*)                             AS total_products,
            ROUND(AVG(price_gbp), 2)             AS avg_price,
            ROUND(MIN(price_gbp), 2)             AS min_price,
            ROUND(MAX(price_gbp), 2)             AS max_price,
            ROUND(AVG(rating_num), 2)            AS avg_rating,
            SUM(in_stock)                        AS in_stock_count,
            ROUND(SUM(in_stock)/COUNT(*)*100, 1) AS availability_pct
        FROM products GROUP BY price_tier
    """,
    "data/tableau_rating_summary.csv": """
        SELECT
            rating_num AS stars,
            COUNT(*) AS count,
            ROUND(AVG(price_gbp), 2) AS avg_price,
            ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct_of_catalogue
        FROM products GROUP BY rating_num ORDER BY rating_num DESC
    """,
}

conn = get_conn(database=DATABASE_NAME)
for path, sql in tableau_queries.items():
    df_t = pd.read_sql(sql, conn)
    df_t.to_csv(path, index=False)
    print(f"✅ {len(df_t):>4} rows → {path}")
conn.close()

print("\n📊 Tableau CSVs ready!")
print("Or connect Tableau directly: Server=localhost | DB=ecommerce_db | Table=products")

---
## 🎉 Project Complete — Deliverables Summary

| File | Description |
|------|-------------|
| `data/raw_products.csv` | Raw scraped data from Selenium |
| `data/clean_products.csv` | Cleaned, transformed dataset |
| `data/excel_report.xlsx` | Formatted Excel workbook (4 sheets, chart, conditional formatting) |
| `data/query_results.xlsx` | All 8 SQL business query results |
| `data/tableau_products.csv` | Full product list for Tableau |
| `data/tableau_tier_summary.csv` | Aggregated tier data for Tableau |
| `data/tableau_rating_summary.csv` | Rating distribution for Tableau |

---

### Tableau Dashboard — Recommended Views

Once connected to `tableau_products.csv` (or directly to MySQL):

1. **Price distribution histogram** — where do most books sit?
2. **Rating vs. price scatter plot** — does price predict quality?
3. **Price tier treemap** — volume and avg rating per tier
4. **Value score bar chart** — top books by rating-to-price ratio

Add a **Rating filter** slider and **Price Tier** selector as interactive controls.

---

### Skills Demonstrated
- Web scraping with **Selenium** (explicit waits, multi-page navigation)
- Data wrangling with **pandas** (type conversion, binning, deduplication)
- **MySQL** schema design, bulk loading, and analytical window functions
- **Excel** automation with openpyxl (pivot tables, charts, conditional formatting)
- End-to-end pipeline from raw HTML → production-ready **Tableau** dashboard
